In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import sys

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rest_of_season.config import (
    SEASON,
    CURRENT_WEEK,
    N_SIMULATIONS,
    RANDOM_SEED,
    CURRENT_OUTPUT_DIR
)

TARGET_WEEK = CURRENT_WEEK

WEEKLY_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed" / "weekly"
)

OUTPUT_DIR = CURRENT_OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Season:", SEASON)
print("Projection week:", TARGET_WEEK)
print("Simulations:", N_SIMULATIONS)
print("Project root:", PROJECT_ROOT)
print("ROS output:", OUTPUT_DIR)


Season: 2026
Projection week: 11
Simulations: 50,000
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_11
Season: 2026
Projection week: 11
Simulations: 50000
Project root: C:\Users\efriedman\Desktop\NFL-Season-Projections
ROS output: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_11


In [2]:

schedule_path = (
    WEEKLY_DATA_DIR / f"{SEASON}_schedule_current.parquet"
)
completed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)
strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)
injury_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

required_files = [
    schedule_path,
    completed_path,
    strength_path,
    injury_path
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required pipeline output missing: {path}"
        )

schedule = pd.read_parquet(schedule_path)
completed = pd.read_parquet(completed_path)
strength = pd.read_parquet(strength_path)
injury_strength = pd.read_parquet(injury_path)

print("Schedule:", schedule.shape)
print("Completed:", completed.shape)
print("Strength:", strength.shape)
print("Injury strength:", injury_strength.shape)


Schedule: (272, 46)
Completed: (150, 46)
Strength: (32, 25)
Injury strength: (32, 30)


In [3]:

remaining_games = schedule.loc[
    (schedule["week"] >= TARGET_WEEK)
    & schedule["home_score"].isna()
    & schedule["away_score"].isna()
].copy()

remaining_games = remaining_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert remaining_games["game_id"].is_unique
assert remaining_games["home_team"].notna().all()
assert remaining_games["away_team"].notna().all()

print("Remaining regular-season games:", len(remaining_games))

display(
    remaining_games[
        ["week", "gameday", "away_team", "home_team"]
    ].head(20)
)


Remaining regular-season games: 122


,week,gameday,away_team,home_team
0,11,2026-11-19,IND,HOU
1,11,2026-11-22,ARI,KC
2,11,2026-11-22,BAL,CAR
3,11,2026-11-22,JAX,NYG
4,11,2026-11-22,LV,DEN
5,11,2026-11-22,MIA,BUF
6,11,2026-11-22,MIN,SF
7,11,2026-11-22,NO,CHI
8,11,2026-11-22,NYJ,LAC
9,11,2026-11-22,PIT,PHI


In [4]:

completed = completed.loc[
    (completed["week"] < TARGET_WEEK)
    & completed["home_score"].notna()
    & completed["away_score"].notna()
].copy()

home_records = completed[
    ["home_team", "home_score", "away_score"]
].rename(columns={
    "home_team": "team",
    "home_score": "points_for",
    "away_score": "points_against"
})

away_records = completed[
    ["away_team", "away_score", "home_score"]
].rename(columns={
    "away_team": "team",
    "away_score": "points_for",
    "home_score": "points_against"
})

team_results = pd.concat(
    [home_records, away_records],
    ignore_index=True
)

team_results["wins"] = (
    team_results["points_for"] > team_results["points_against"]
).astype(int)

team_results["losses"] = (
    team_results["points_for"] < team_results["points_against"]
).astype(int)

team_results["ties"] = (
    team_results["points_for"] == team_results["points_against"]
).astype(int)

records = team_results.groupby("team", as_index=False).agg(
    wins=("wins", "sum"),
    losses=("losses", "sum"),
    ties=("ties", "sum"),
    games_played=("wins", "size"),
    points_for=("points_for", "sum"),
    points_against=("points_against", "sum")
)

records["point_differential"] = (
    records["points_for"] - records["points_against"]
)

display(records.sort_values(
    ["wins", "point_differential"],
    ascending=[False, False]
))


,team,wins,losses,ties,games_played,points_for,points_against,point_differential
3,BUF,8,1,0,9,287.0,199.0,88.0
28,SF,8,1,0,9,281.0,196.0,85.0
14,JAX,7,2,0,9,269.0,144.0,125.0
6,CIN,7,2,0,9,278.0,193.0,85.0
15,KC,7,2,0,9,237.0,156.0,81.0
10,DET,7,2,0,9,257.0,199.0,58.0
5,CHI,7,2,0,9,241.0,184.0,57.0
20,MIN,7,2,0,9,176.0,126.0,50.0
9,DEN,7,2,0,9,198.0,186.0,12.0
16,LA,6,4,0,10,248.0,187.0,61.0


In [5]:

required_strength_cols = [
    "team",
    "weekly_team_strength"
]

required_injury_cols = [
    "team",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

for name, frame, columns in [
    ("strength", strength, required_strength_cols),
    ("injury", injury_strength, required_injury_cols)
]:
    missing = set(columns) - set(frame.columns)
    if missing:
        raise KeyError(
            f"{name} missing columns: {sorted(missing)}"
        )

team_snapshot = strength[
    required_strength_cols
].merge(
    injury_strength[required_injury_cols],
    on="team",
    how="left",
    validate="one_to_one"
).merge(
    records,
    on="team",
    how="left",
    validate="one_to_one"
)

assert len(team_snapshot) == 32
assert team_snapshot["team"].is_unique
assert team_snapshot["weekly_team_strength"].notna().all()

print("Teams:", len(team_snapshot))
display(team_snapshot.head(10))


Teams: 32


,team,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength,wins,losses,ties,games_played,points_for,points_against,point_differential
0,BUF,8.151564,-0.270675,7.880889,8,1,0,9,287.0,199.0,88.0
1,JAX,6.485660,-0.848250,5.637410,7,2,0,9,269.0,144.0,125.0
2,LA,5.981859,-1.484025,4.497834,6,4,0,10,248.0,187.0,61.0
3,SF,5.208822,-0.669300,4.539522,8,1,0,9,281.0,196.0,85.0
4,KC,5.070234,-0.646875,4.423359,7,2,0,9,237.0,156.0,81.0
5,DET,4.974895,-0.932100,4.042795,7,2,0,9,257.0,199.0,58.0
6,SEA,4.310075,-0.253875,4.056200,6,4,0,10,223.0,175.0,48.0
7,CIN,4.292347,-0.257850,4.034497,7,2,0,9,278.0,193.0,85.0
8,DEN,3.953856,-0.594975,3.358881,7,2,0,9,198.0,186.0,12.0
9,CHI,3.472897,-0.969300,2.503597,7,2,0,9,241.0,184.0,57.0


In [6]:

# Validate Week 9 starting standings and team snapshots.
# Read-only; does not overwrite historical files.

standings_check = team_snapshot.copy()

standings_check["record"] = (
    standings_check["wins"].astype(int).astype(str)
    + "-"
    + standings_check["losses"].astype(int).astype(str)
    + "-"
    + standings_check["ties"].astype(int).astype(str)
)

standings_check["remaining_games"] = (
    standings_check["team"].map(
        pd.concat([
            remaining_games["home_team"],
            remaining_games["away_team"]
        ]).value_counts()
    ).fillna(0).astype(int)
)

standings_check["season_games"] = (
    standings_check["games_played"]
    + standings_check["remaining_games"]
)

# Validate all teams and schedules
assert len(standings_check) == 32

assert standings_check["season_games"].eq(17).all(), (
    "At least one team does not have 17 total games."
)

assert (
    standings_check["wins"]
    + standings_check["losses"]
    + standings_check["ties"]
).eq(standings_check["games_played"]).all(), (
    "Wins, losses, and ties do not match games played."
)

assert standings_check[
    "injury_adjusted_team_strength"
].notna().all(), (
    "Missing injury-adjusted team strength."
)

# Sort BEFORE selecting display columns
standings_check = (
    standings_check
    .sort_values(
        ["wins", "ties", "point_differential"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print("Current NFL standings entering Week 9:")

display(
    standings_check[
        [
            "team",
            "record",
            "games_played",
            "remaining_games",
            "point_differential",
            "weekly_team_strength",
            "injury_adjustment",
            "injury_adjusted_team_strength"
        ]
    ]
)

print("\nAll 32 teams have 17 scheduled games.")
print("Starting standings and team ratings validated.")


Current NFL standings entering Week 9:


,team,record,games_played,remaining_games,point_differential,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength
0,BUF,8-1-0,9,8,88.0,8.151564,-0.270675,7.880889
1,SF,8-1-0,9,8,85.0,5.208822,-0.669300,4.539522
2,JAX,7-2-0,9,8,125.0,6.485660,-0.848250,5.637410
3,CIN,7-2-0,9,8,85.0,4.292347,-0.257850,4.034497
4,KC,7-2-0,9,8,81.0,5.070234,-0.646875,4.423359
5,DET,7-2-0,9,8,58.0,4.974895,-0.932100,4.042795
6,CHI,7-2-0,9,8,57.0,3.472897,-0.969300,2.503597
7,MIN,7-2-0,9,8,50.0,3.069441,-0.225075,2.844366
8,DEN,7-2-0,9,8,12.0,3.953856,-0.594975,3.358881
9,LA,6-4-0,10,7,61.0,5.981859,-1.484025,4.497834



All 32 teams have 17 scheduled games.
Starting standings and team ratings validated.


In [7]:

team_snapshot["season"] = SEASON
team_snapshot["snapshot_week"] = TARGET_WEEK

remaining_games["season"] = SEASON
remaining_games["snapshot_week"] = TARGET_WEEK

team_snapshot.to_parquet(
    OUTPUT_DIR / "team_snapshot.parquet",
    index=False
)

remaining_games.to_parquet(
    OUTPUT_DIR / "remaining_schedule.parquet",
    index=False
)

team_snapshot.to_csv(
    OUTPUT_DIR / "team_snapshot.csv",
    index=False
)

remaining_games[
    ["week", "gameday", "away_team", "home_team"]
].to_csv(
    OUTPUT_DIR / "remaining_schedule.csv",
    index=False
)

print(f"Saved Week {TARGET_WEEK} ROS inputs.")
print("Output directory:", OUTPUT_DIR)
print("Teams:", len(team_snapshot))
print("Remaining games:", len(remaining_games))


Saved Week 11 ROS inputs.
Output directory: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_11
Teams: 32
Remaining games: 122
